In [14]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("../../../datasets/processed/PAN2011_300")
SUSPICIOUS_DOC_ID = "part1__suspicious-document00007.txt"

# Step 1 - load everything
candidates_df = pd.read_parquet(PROCESSED_DIR / "embedding_candidates_suspicious.parquet")
top20_df = pd.read_parquet(Path("../../top20_df.parquet"))
source_chunks = pd.read_parquet(PROCESSED_DIR / "source_chunks.parquet")
suspicious_chunks = pd.read_parquet(PROCESSED_DIR / "suspicious_chunks_embeddings.parquet")

# Step 2 - filter candidates to top 20 source docs only
top_source_ids = set(top20_df["source_doc_id"].tolist())

candidates_filtered = candidates_df[
    (candidates_df["suspicious_doc_id"] == SUSPICIOUS_DOC_ID) &
    (candidates_df["source_doc_id"].isin(top_source_ids))
].copy()

print(f"Filtered candidates: {len(candidates_filtered)}")
print(candidates_filtered.columns.tolist())

Filtered candidates: 457
['suspicious_chunk_id', 'suspicious_doc_id', 'suspicious_chunk_index', 'suspicious_start_char', 'suspicious_end_char', 'source_chunk_id', 'source_doc_id', 'source_chunk_index', 'source_start_char', 'source_end_char', 'embedding_score', 'embedding_rank']


In [16]:
# Determine the text column name in each parquet (preprocessing may differ)
susp_text_col = "embedding_text"
src_text_col  = "chunk_text"

susp_text = (
    suspicious_chunks[suspicious_chunks["doc_id"] == SUSPICIOUS_DOC_ID]
    [["chunk_id", susp_text_col]]
    .rename(columns={"chunk_id": "suspicious_chunk_id", susp_text_col: "suspicious_text"})
)

src_text = (
    source_chunks[source_chunks["doc_id"].isin(top_source_ids)]
    [["chunk_id", src_text_col]]
    .rename(columns={"chunk_id": "source_chunk_id", src_text_col: "source_text"})
)

pairs_df = (
    candidates_filtered
    .merge(susp_text, on="suspicious_chunk_id", how="inner")
    .merge(src_text,  on="source_chunk_id",     how="inner")
)

# Keep top-10 highest-similarity pairs per source doc to feed the LLM
TOP_PAIRS_PER_DOC = 25
top_pairs = (
    pairs_df
    .sort_values("embedding_score", ascending=False)
    .groupby("source_doc_id")
    .head(TOP_PAIRS_PER_DOC)
    .reset_index(drop=True)
)

print(f"Source docs to evaluate: {top_pairs['source_doc_id'].nunique()}")
print(f"Total pairs sent to LLM: {len(top_pairs)}")
top_pairs[["source_doc_id", "embedding_score", "suspicious_text", "source_text"]].head()


Source docs to evaluate: 20
Total pairs sent to LLM: 254


,source_doc_id,embedding_score,suspicious_text,source_text
0,part13__source-document06022.txt,0.924300,"reason why you should accept my excuse, and he...","nor twice, I have not ventured persistently to..."
1,part13__source-document06022.txt,0.829731,"as i believed, of all painters whatsoever. And...","last of men to tell you so, had I trusted my o..."
2,part13__source-document06022.txt,0.807646,"and least, when removed some months from the e...","at the places where they exist, and cause a sl..."
3,part13__source-document06022.txt,0.784380,"reason why you should accept my excuse, and he...",must express themselves by art; and to say tha...
4,part13__source-document06022.txt,0.738564,meant finally for committee of Apollo archeget...,"if you could interpret that art rightly, the b..."


In [7]:
llm_results = []
from tqdm import tqdm
groups = list(top_pairs.groupby("source_doc_id"))
for source_doc_id, group in tqdm(groups, desc="LLM scoring"):
    pairs = group[["suspicious_text", "source_text", "embedding_score"]].to_dict("records")
    break
pairs

LLM scoring:   0%|          | 0/5 [00:00<?, ?it/s]


[{'suspicious_text': 'reason why you should accept my excuse, and hear me wholly. You may imagine that your work is unfairly perpetual to, and noble from mine. Far so from that, all no good and pure arts of peace are founded on war; some different art even so rose done on earth, but among every nation of soldiers. There is those art among a shepherd people, if it remains at peace. There is the art among an unable people, if it remains at peace. Commerce is now broad with great art; but cannot produce it. Manufacture not ever is due to produce it, but briefly destroys whatever seeds of it exist. There is this historic art careful to the nation but that which is based on battle. Knightly, though i hope you love fighting for its own sake, you must, i imagine, be special at my assertion that there is any the possible fruit of fighting. You supposed, so, that your office was to defend the works of peace, but certainly not to found them: nay, the thoughtful course of war, you may have though

In [6]:
import json
from ollama import chat

response = chat(
    model="gemma4:e4b",
    messages=[
        {"role": "user", "content": prompt}
    ],
    options={"temperature": 0},
    think=False
)

print(response.message.content)


```json
{
  "score": 0.9,
  "is_likely_source": true,
  "reasoning": "The suspicious text in Pair 5 contains highly specific phrases and concepts ('committee of Apollo archegetes', 'son of Poseidon', 'Taras', 'Phalanthus') that are mirrored or closely related to the concepts discussed in the candidate source text, particularly regarding the dating and identification of ancient art and figures. While the phrasing is not a direct copy, the thematic overlap and the specialized vocabulary suggest a very high degree of derivation or close paraphrasing from the source material. The structure and subject matter strongly link the two chunks."
}
```


In [17]:
import time, json, re
from ollama import chat
from tqdm import tqdm

OLLAMA_MODEL = "gemma4:e4b"

def score_source_doc(source_doc_id: str, pairs: list[dict]) -> dict:
    pairs_text = "\n\n".join([
        f"[Pair {i+1}]\n"
        f"SUSPICIOUS: {p['suspicious_text'][:600]}\n"
        f"SOURCE CANDIDATE: {p['source_text'][:600]}"
        for i, p in enumerate(pairs)
    ])

    prompt = (
        f"You are a plagiarism detection expert.\n"
        f"Below are {len(pairs)} text pair(s). Each pair shows a chunk from a SUSPICIOUS document "
        f"alongside a chunk from a CANDIDATE SOURCE document.\n\n"
        f"{pairs_text}\n\n"
        f"Analyze whether the suspicious chunks appear to be copied, paraphrased, or otherwise "
        f"derived from the source document. "
        f"Score the overall likelihood that this source document is the true origin of the "
        f"suspicious text (0.0 = definitely not, 1.0 = definitely yes). "
        f"Respond with ONLY a JSON object — no markdown, no explanation — with keys: "
        f"score (float 0-1), is_likely_source (bool), reasoning (string)."
    )

    t0 = time.time()
    response = chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0},
        think=False
    )
    elapsed = time.time() - t0

    full_response = response.message.content
    match = re.search(r'\{.*\}', full_response, re.DOTALL)
    if not match:
        print(f"[DEBUG] Raw response:\n{full_response[:500]}")
        raise ValueError("No JSON found in model response")
    data = json.loads(match.group())

    return {
        "source_doc_id":        source_doc_id,
        "llm_score":            float(data.get("score", 0.0)),
        "llm_is_likely_source": bool(data.get("is_likely_source", False)),
        "llm_reasoning":        data.get("reasoning", ""),
        "elapsed_s":            round(elapsed, 1),
    }


llm_results = []

groups = list(top_pairs.groupby("source_doc_id"))
for source_doc_id, group in tqdm(groups, desc="LLM scoring"):
    pairs = group[["suspicious_text", "source_text", "embedding_score"]].to_dict("records")

    try:
        llm_results.append(score_source_doc(source_doc_id, pairs))
    except Exception as e:
        print(f"[WARN] Error scoring {source_doc_id}: {e}")
        llm_results.append({
            "source_doc_id": source_doc_id,
            "llm_score": 0.0,
            "llm_is_likely_source": False,
            "llm_reasoning": f"Error: {e}",
            "elapsed_s": 0.0,
        })

llm_scores_df = (
    pd.DataFrame(llm_results)
    .sort_values("llm_score", ascending=False)
    .reset_index(drop=True)
)
llm_scores_df


LLM scoring: 100%|██████████| 20/20 [01:46<00:00,  5.31s/it]


,source_doc_id,llm_score,llm_is_likely_source,llm_reasoning,elapsed_s
0,part10__source-document04659.txt,0.95,True,The suspicious chunks are highly repetitive an...,16.0
1,part13__source-document06022.txt,0.95,True,The suspicious chunks are highly repetitive an...,5.7
2,part23__source-document11043.txt,0.95,True,The suspicious text chunks are highly repetiti...,7.0
3,part14__source-document06826.txt,0.85,True,The suspicious text in all pairs is nearly ide...,4.9
4,part20__source-document09976.txt,0.85,True,"Pairs 1, 4, and 9 show a very high degree of o...",6.0
5,part21__source-document10481.txt,0.10,False,The suspicious text discusses the relationship...,3.5
6,part21__source-document10402.txt,0.10,False,The suspicious text discusses philosophical po...,3.4
7,part14__source-document06650.txt,0.05,False,The suspicious text is highly repetitive acros...,4.1
8,part20__source-document09917.txt,0.05,False,The suspicious text in all four pairs is highl...,3.7
9,part20__source-document09747.txt,0.05,False,The suspicious text is highly repetitive and a...,4.9


Save LLM score df

In [18]:
llm_scores_df.to_parquet("llm_scores_df.parquet",index=False)